In [1]:
print("hello world")

hello world


In [2]:
# Test-1

import yaml
from dotenv import load_dotenv
import os

import warnings
warnings.filterwarnings('ignore')

# Load environment variables
load_dotenv()

# Load config.yaml
with open("config.yaml", "r") as f:
    config = yaml.safe_load(f)

print("✅ Environment loaded successfully")
print("✅ LLM Provider:", config["llm"]["provider"])
print("✅ OpenAI Key exists:", "OPENAI_API_KEY" in os.environ)
print("✅ Google Key exists:", "GOOGLE_API_KEY" in os.environ)

✅ Environment loaded successfully
✅ LLM Provider: openai
✅ OpenAI Key exists: True
✅ Google Key exists: True


In [3]:
from utils.loader import load_and_chunk_docs
from utils.retriever_bm25 import BM25Retriever

chunks = load_and_chunk_docs("./data/raw/insurance_docs")

bm25 = BM25Retriever(chunks) #Object of BM25Retriever class is created with the loaded document chunks

query = "Essential Documents"
results = bm25.get_top_k(query, k=5) #results is a list of tuples where each tuple contains a Document object 
#and its corresponding BM25 score

print("🔍 BM25 Results:\n")
for i, (doc, score) in enumerate(results):
    print(f"---- Result {i+1} (score={score:.2f}) ----") #upto 2 decimal places for score
    print(doc.page_content[:200], "\n") #print the first 200 characters of the document content

✅ Loaded 2 docs → 8 chunks
BM25 Scores: [0.         3.15058284 0.         0.         0.         0.
 0.         0.        ]
🔍 BM25 Results:

---- Result 1 (score=3.15) ----
Contact information for any third parties or witnesses.

II. Documentation Submission

After initiating your claim (Status: SUBMITTED), you will receive a Claim ID (e.g., CLM-2025-1010). Please upload 

---- Result 2 (score=0.00) ----
Replacement Cost

The cost to replace a damaged item with a brand new one of similar kind and quality, without deduction for depreciation. 

---- Result 3 (score=0.00) ----
Coverage Limit

The maximum amount the insurance company will pay for a covered loss, as stated in the policy schedule.

Exclusion

A specific event, property, or type of loss that is not covered by t 

---- Result 4 (score=0.00) ----
Effective Date

The date and time when coverage under the policy officially begins.

Deductible

The specific amount of money you must pay out-of-pocket before the insurer begins to cov

In [11]:
# Test-3

from utils.loader import load_and_chunk_docs
from utils.retriever_faiss import get_retriever
import yaml

config = yaml.safe_load(open("config.yaml"))
chunks = load_and_chunk_docs("./data/raw/insurance_docs")

retriever = get_retriever(config, chunks_if_needed=chunks)
print("✅ Retriever ready. Sample:", type(retriever))

✅ Loaded 2 docs → 8 chunks
✅ Loaded FAISS index (openai) from: ./data/embeddings/faiss_openai
✅ Retriever ready. Sample: <class 'langchain_core.vectorstores.base.VectorStoreRetriever'>


In [5]:
# Test-4

from utils.loader import load_and_chunk_docs
from utils.hybrid_retriever import HybridRetriever

chunks = load_and_chunk_docs("./data/raw/insurance_docs")

hybrid = HybridRetriever(chunks)

query = "cashless hospitalization"

results = hybrid.retrieve(query, top_k=3) 
#results is a list of tuples where each tuple contains a Document object and its corresponding hybrid score

print("\n🔍 HYBRID SEARCH RESULTS:\n")
for i, (doc, score) in enumerate(results):
    print(f"--- Result {i+1} (score={score:.4f}) ---") #score is printed upto 4 decimal places
    print(doc.page_content[:250], "\n") #print the first 250 characters of the document content

✅ Loaded 2 docs → 8 chunks
✅ Loaded FAISS index (openai) from: ./data/embeddings/faiss_openai
BM25 Scores: [0. 0. 0. 0. 0. 0. 0. 0.]

🔍 HYBRID SEARCH RESULTS:

--- Result 1 (score=0.6000) ---
Medical records/bills (for injury claims).

III. Review and Assessment

Status: PROCESSING
A dedicated Claims Adjuster will be assigned to your case. They will review all documentation, verify coverage against your policy terms, and may schedule a ph 

--- Result 2 (score=0.4500) ---
Coverage Limit

The maximum amount the insurance company will pay for a covered loss, as stated in the policy schedule.

Exclusion

A specific event, property, or type of loss that is not covered by the insurance policy. Common exclusions include act 

--- Result 3 (score=0.4000) ---
IV. Final Decision and Settlement

The adjuster will issue a final determination:

APPROVED: Your claim meets the coverage criteria. A settlement offer will be issued, or payment will be made directly to the service provider. (Status: APP